# Credit Default Prediction: Baseline and History Comparison

We compare two Logistic Regression models: a five-feature course baseline and a 23-feature model that adds payment history. Both use an 80/20 stratified split (seed 42). Hyperparameters are selected using stratified five-fold CV inside the training data only.

**Evaluation:** mean CV Log Loss is the primary selection metric (lower is better); ROC-AUC measures ranking performance. Tables report fold SD and SE, with SE = sample SD / sqrt(5). These are descriptive error bars across overlapping CV folds, not independent-sample confidence intervals. CV results after selecting C may be optimistic.

The original notebook already displayed holdout results. The test results below are therefore exploratory; prior exposure cannot be undone. Neither C nor the fixed classification threshold (0.50) is selected using the test set.

## 1. Load Data

In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split

file_path = "credit_flag1_kept.csv"
df = pd.read_csv(file_path)

print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

Dataset shape: 27,596 rows x 27 columns


Data source: `credit_flag1_kept.csv`. Use the supplied rows directly; exclude `ID`, `flag_1`, and `flag_2` from predictors to retain the original 23-feature comparison.

## 2. Train/Test Split

In [32]:
target_col = "default payment next month"

X = df.drop(columns=["ID", target_col, "flag_1", "flag_2"])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)
split_summary = pd.DataFrame(
    {
        "Samples": [len(y_train), len(y_test)],
        "Features": [X_train.shape[1], X_test.shape[1]],
        "Defaults": [y_train.sum(), y_test.sum()],
        "Default Rate": [f"{y_train.mean():.3%}", f"{y_test.mean():.3%}"],
    },
    index=["Train", "Test"],
)

split_summary

,Samples,Features,Defaults,Default Rate
Train,22076,23,4796,21.725%
Test,5520,23,1199,21.721%


## 3. Feature Sets and Preprocessing

**Baseline (5):** LIMIT_BAL, SEX, EDUCATION, MARRIAGE, AGE.  
**History (23):** the baseline plus six repayment-status, six bill-amount, and six payment-amount variables.

Numeric variables are standardized; SEX, EDUCATION, MARRIAGE, and repayment status are one-hot encoded. Each preprocessing step is fitted inside the CV training fold.

In [33]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

baseline_features = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"]
status_features = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

feature_sets = {
    "Baseline (5)": baseline_features,
    "History (23)": X.columns.tolist(),
}
categorical_features = ["SEX", "EDUCATION", "MARRIAGE"] + status_features

models = {}
for name, features in feature_sets.items():
    categorical = [col for col in features if col in categorical_features]
    numeric = [col for col in features if col not in categorical_features]
    preprocess = ColumnTransformer([
        ("numeric", StandardScaler(), numeric),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical),
    ])
    models[name] = Pipeline([
        ("preprocess", preprocess),
        ("logistic", LogisticRegression(max_iter=2000, solver="lbfgs")),
    ])

## 4. Tune Both Models with Five-Fold CV

Search C in [0.01, 0.1, 1, 10] (smaller C means stronger regularization). Each model selects its own C by minimum mean CV Log Loss. The same validation folds are used for both feature sets.

In [34]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

param_grid = {"logistic__C": [0.01, 0.1, 1.0, 10.0]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_splits = list(cv.split(X_train, y_train))
scoring = {
    "log_loss": "neg_log_loss",
    "roc_auc": "roc_auc",
}

searches = {}
for name, model in models.items():
    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=scoring,
        cv=cv_splits,
        refit=False,
        error_score="raise",
    )
    search.fit(X_train[feature_sets[name]], y_train)
    searches[name] = search

## 5. CV Results and Error Bars

In [35]:
cv_tables = []
best_params = {}
best_indices = {}

for name, search in searches.items():
    results = pd.DataFrame(search.cv_results_)
    best_index = results["mean_test_log_loss"].idxmax()
    best_indices[name] = best_index
    best_params[name] = results.loc[best_index, "params"]
    cv_tables.append(pd.DataFrame({
        "Model": name,
        "C": results["param_logistic__C"].astype(float),
        "Mean Log Loss": -results["mean_test_log_loss"],
        "Mean ROC-AUC": results["mean_test_roc_auc"],
        "Selected": results.index == best_index,
    }))

cv_summary = pd.concat(cv_tables, ignore_index=True)
display(cv_summary.round(4))

,Model,C,Mean Log Loss,Mean ROC-AUC,Selected
0,Baseline (5),0.01,0.5060,0.6343,False
1,Baseline (5),0.10,0.5053,0.6356,False
2,Baseline (5),1.00,0.5052,0.6357,True
3,Baseline (5),10.00,0.5052,0.6357,False
4,History (23),0.01,0.4270,0.7756,False
5,History (23),0.10,0.4247,0.7760,True
6,History (23),1.00,0.4248,0.7759,False
7,History (23),10.00,0.4251,0.7756,False


In [36]:
import numpy as np

cv_rows = []
fold_losses = {}

for name, search in searches.items():
    results = search.cv_results_
    index = best_indices[name]
    row = {"Model": name, "Best C": best_params[name]["logistic__C"]}
    for metric, label in [
        ("log_loss", "Log Loss"),
        ("roc_auc", "ROC-AUC"),
    ]:
        scores = np.array([results[f"split{i}_test_{metric}"][index] for i in range(5)])
        if metric == "log_loss":
            scores = -scores
            fold_losses[name] = scores
        row[f"{label} Mean"] = scores.mean()
        row[f"{label} SD"] = scores.std(ddof=1)
        row[f"{label} SE"] = scores.std(ddof=1) / np.sqrt(5)
    cv_rows.append(row)

cv_comparison = pd.DataFrame(cv_rows).set_index("Model")
display(cv_comparison.round(4))

,Best C,Log Loss Mean,Log Loss SD,Log Loss SE,ROC-AUC Mean,ROC-AUC SD,ROC-AUC SE
Model,,,,,,,
Baseline (5),1.0,0.5052,0.0033,0.0015,0.6357,0.0080,0.0036
History (23),0.1,0.4247,0.0097,0.0044,0.7760,0.0129,0.0058


**Table 1. Preliminary results for the five-feature baseline and 23-feature history model.** Each row uses that model's selected C. The Mean and SE columns specify a mean +/- one SE error bar for each metric; SD uses ddof=1. Lower Log Loss and higher ROC-AUC are better.

In [37]:
paired_reduction = fold_losses["Baseline (5)"] - fold_losses["History (23)"]
paired_results = pd.DataFrame({
    "Baseline Log Loss": fold_losses["Baseline (5)"],
    "History Log Loss": fold_losses["History (23)"],
    "Reduction": paired_reduction,
}, index=pd.Index(range(1, 6), name="Fold"))
display(paired_results.round(4))

paired_summary = pd.Series({
    "Mean Reduction": paired_reduction.mean(),
    "Reduction SD": paired_reduction.std(ddof=1),
    "Reduction SE": paired_reduction.std(ddof=1) / np.sqrt(5),
}, name="Paired CV Comparison")
display(paired_summary.round(4).to_frame())

,Baseline Log Loss,History Log Loss,Reduction
Fold,,,
1,0.5059,0.4246,0.0813
2,0.5099,0.4407,0.0692
3,0.5041,0.4163,0.0879
4,0.5053,0.4245,0.0808
5,0.5007,0.4175,0.0832


,Paired CV Comparison
Mean Reduction,0.0805
Reduction SD,0.0069
Reduction SE,0.0031


**Table 2. Paired improvement from history features.** Reduction = baseline Log Loss - history Log Loss on the same fold. Positive values favor history. Mean Reduction +/- Reduction SE is a descriptive error bar, not a formal significance test.

## 6. Fit Final Models and Predict the Test Set

In [38]:
from sklearn.base import clone

final_models = {}
test_probabilities = {}
test_predictions = {}
threshold = 0.50

for name, model in models.items():
    final_model = clone(model).set_params(**best_params[name])
    final_model.fit(X_train[feature_sets[name]], y_train)
    probabilities = final_model.predict_proba(X_test[feature_sets[name]])[:, 1]
    final_models[name] = final_model
    test_probabilities[name] = probabilities
    test_predictions[name] = (probabilities >= threshold).astype(int)

## 7. Test Metrics and Confusion Matrices

Log Loss evaluates probabilities (lower is better). ROC-AUC evaluates ranking. Accuracy, Precision, Recall, and F1 evaluate decisions at threshold 0.50. Recall is the fraction of actual defaults detected. Class imbalance makes Accuracy alone insufficient.

In [39]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)

test_rows = []
confusion_matrices = {}

for name in models:
    probabilities = test_probabilities[name]
    predictions = test_predictions[name]
    matrix = confusion_matrix(y_test, predictions, labels=[0, 1])
    test_rows.append({
        "Model": name,
        "Log Loss": log_loss(y_test, probabilities),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
    })
    confusion_matrices[name] = pd.DataFrame(
        matrix,
        index=["Actual Non-Default", "Actual Default"],
        columns=["Predicted Non-Default", "Predicted Default"],
    )

test_metrics = pd.DataFrame(test_rows).set_index("Model")
display(test_metrics.T.round(4))

Model,Baseline (5),History (23)
Log Loss,0.5083,0.4376
ROC-AUC,0.6211,0.7608
Accuracy,0.7828,0.8168
Precision,0.0000,0.6407
Recall,0.0000,0.3570
F1,0.0000,0.4585


**Table 3. Exploratory holdout performance of both models.** Each model is fitted on all training data with its CV-selected C. Classification metrics use threshold 0.50; probability and ranking metrics use predicted probabilities. These test scores must not be used for further tuning.

In [40]:
for name, matrix in confusion_matrices.items():
    print(name)
    display(matrix)

Baseline (5)


,Predicted Non-Default,Predicted Default
Actual Non-Default,4321,0
Actual Default,1199,0


History (23)


,Predicted Non-Default,Predicted Default
Actual Non-Default,4081,240
Actual Default,771,428
